# Crowd Predictor Framework - Integrated Model Training
**Status:** Results injected from automated background run.


In [2]:
import pandas as pd
import os
import sys
import torch
import pickle
import numpy as np
from sklearn.metrics import mean_absolute_error
from prophet import Prophet
from prophet.serialize import model_to_json
import logging
import warnings

# --- MONKEYPATCH: Fix CmdStanPy/Polars TypeError ---
try:
    import cmdstanpy.utils.stancsv as stancsv
    def stable_csv_to_numpy(csv_bytes_list):
        if not csv_bytes_list:
            return np.empty((0, 0))
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            out = np.loadtxt(csv_bytes_list, delimiter=",", dtype=np.float64, ndmin=2)
        return out if out.shape[0] > 0 else np.empty((0, 0))
    
    stancsv.csv_bytes_list_to_numpy = stable_csv_to_numpy
    print('[OK] Monkeypatch applied: CmdStanPy now using stable NumPy parser.')
except Exception as e:
    print(f'[WARN] Monkeypatch failed: {e}')

# --- Logic ---
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.ml.lstm_model import CrowdLSTMModel

# Data Loading
print('--- Loading Dataset ---')
df = pd.read_csv('data/crowd_data.csv')
df['ds'] = pd.to_datetime(df['timestamp'])
df['y'] = df['count'].rolling(window=3, min_periods=1).mean()
df = df.dropna(subset=['y']).sort_values('ds').reset_index(drop=True)
df['hour'] = df['ds'].dt.hour
df['day'] = df['ds'].dt.dayofweek

train_size = int(len(df) * 0.8)
df_train_full = df.iloc[:train_size].copy()
df_test_full  = df.iloc[train_size:].copy()
df_recent = df.tail(10000).copy()
recent_split = int(len(df_recent) * 0.8)
df_train_lstm = df_recent.iloc[:recent_split].copy()
df_test_lstm  = df_recent.iloc[recent_split:].copy()

# Training Prophet
print('\n--- Training Prophet (Stable Mode) ---')
prophet_model = Prophet(changepoint_prior_scale=0.1, daily_seasonality=True, weekly_seasonality=True)
prophet_model.add_regressor('hour')
prophet_model.add_regressor('day')
prophet_model.fit(df_train_full[['ds', 'y', 'hour', 'day']])
with open('models/prophet_model.json', 'w') as f:
    f.write(model_to_json(prophet_model))
print('[OK] Prophet saved.')

# Training LSTM
print('\n--- Training LSTM ---')
lstm = CrowdLSTMModel(sequence_length=20, epochs=50, lr=0.001)
lstm.train(df_train_lstm[['ds', 'y', 'hour', 'day']])
lstm.save()
print('[OK] LSTM saved.')

# Evaluation
print('\n--- Final Results ---')
p_future = df_test_full[['ds', 'hour', 'day']].copy()
p_forecast = prophet_model.predict(p_future)
p_pred = np.clip(p_forecast['yhat'].values, 0, None)
p_mae = np.mean(np.abs(df_test_full['y'].values - p_pred))
p_acc = (1 - p_mae / df_test_full['y'].mean()) * 100
print(f'Prophet Accuracy: {p_acc:.2f}%')

lstm.model.eval()
SEQUENCE_LENGTH = 20
test_data_norm = lstm.scaler.transform(df_test_lstm[['y', 'hour', 'day']].values)
X_test = torch.FloatTensor([test_data_norm[i:i+SEQUENCE_LENGTH] for i in range(len(test_data_norm)-SEQUENCE_LENGTH)])
with torch.no_grad():
    preds_norm = lstm.model(X_test).squeeze().numpy()
preds_padded = np.zeros((len(preds_norm), 3))
preds_padded[:, 0] = preds_norm
l_pred = lstm.scaler.inverse_transform(preds_padded)[:, 0]
l_true = df_test_lstm['y'].values[SEQUENCE_LENGTH:]
l_mae = np.mean(np.abs(l_true - l_pred))
l_acc = (1 - l_mae / l_true.mean()) * 100
print(f'LSTM Accuracy:    {l_acc:.2f}%')


[OK] Monkeypatch applied: CmdStanPy now using stable NumPy parser.
--- Loading Dataset ---

--- Training Prophet (Stable Mode) ---
[OK] Prophet saved.

--- Training LSTM ---
[OK] LSTM saved.

--- Final Results ---
Prophet Accuracy: 73.45%


C:\Users\hp\AppData\Local\Temp\ipykernel_22572\1260560341.py:83: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  X_test = torch.FloatTensor([test_data_norm[i:i+SEQUENCE_LENGTH] for i in range(len(test_data_norm)-SEQUENCE_LENGTH)])


LSTM Accuracy:    90.66%


In [7]:
import numpy as np
import torch
import torch.nn as nn
import pickle
import pandas as pd
from sklearn.metrics import mean_absolute_error

# 1. FIXED PARAMETERS (Matching your saved checkpoint)
HIDDEN_SIZE = 128  # Changed from 64 to 128
NUM_LAYERS = 2     # Changed from 1 to 2
DROPOUT = 0.2
SEQUENCE_LENGTH = 24

# 2. MATCHING ARCHITECTURE DEFINITION
class ImprovedLSTMNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(ImprovedLSTMNet, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=dropout)
        # Renamed from self.fc to self.linear to match your saved weights
        self.linear = nn.Linear(hidden_size, 1) 

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.linear(out[:, -1, :])
        return out

print('--- Model Evaluation ---')

# -----------------------------------------------------------------------
# 3. PROPHET EVALUATION
# -----------------------------------------------------------------------
try:
    future = df_test_full[['ds', 'hour', 'day']].copy() 
    forecast = prophet_model.predict(future)
    prophet_y_pred = np.clip(forecast['yhat'].values, 0, None)
    prophet_y_true = df_test_full['y'].values
    prophet_mae  = mean_absolute_error(prophet_y_true, prophet_y_pred)
    prophet_mean = np.mean(prophet_y_true)
    prophet_accuracy = max(0, (1 - (prophet_mae / prophet_mean)) * 100) if prophet_mean != 0 else 0
    print(f'Prophet MAE:      {prophet_mae:.2f}')
    print(f'Prophet Accuracy: {prophet_accuracy:.2f}%')
except Exception as e:
    print(f"Prophet Error: {e}")

# -----------------------------------------------------------------------
# 4. LSTM EVALUATION (FIXED LOAD)
# -----------------------------------------------------------------------
try:
    with open('models/scaler.pkl', 'rb') as f:
        eval_scaler = pickle.load(f)

    features = ['y', 'hour', 'day']
    test_data = df_test_lstm[features].values
    test_data_norm = eval_scaler.transform(test_data)

    X_test_list = []
    for i in range(len(test_data_norm) - SEQUENCE_LENGTH):
        X_test_list.append(test_data_norm[i : i + SEQUENCE_LENGTH])
    X_test_tensor = torch.FloatTensor(np.array(X_test_list))

    # Initialize model with correct 128 hidden size and 2 layers
    eval_net = ImprovedLSTMNet(
        input_size=len(features), 
        hidden_size=HIDDEN_SIZE, 
        num_layers=NUM_LAYERS, 
        dropout=DROPOUT
    )
    
    # Loading weights - strict=True ensures every layer name matches
    eval_net.load_state_dict(torch.load('models/lstm_weights.pth'))
    eval_net.eval()

    with torch.no_grad():
        preds_norm = eval_net(X_test_tensor).squeeze().numpy()

    preds_padded = np.zeros((len(preds_norm), len(features)))
    preds_padded[:, 0] = preds_norm
    lstm_y_pred = eval_scaler.inverse_transform(preds_padded)[:, 0]
    lstm_y_true = test_data[SEQUENCE_LENGTH:, 0]

    lstm_mae      = mean_absolute_error(lstm_y_true, lstm_y_pred)
    lstm_mean     = np.mean(lstm_y_true)
    lstm_accuracy = max(0, (1 - (lstm_mae / lstm_mean)) * 100) if lstm_mean != 0 else 0

    print(f'LSTM MAE:         {lstm_mae:.2f}')
    print(f'LSTM Accuracy:    {lstm_accuracy:.2f}%')
except Exception as e:
    print(f"LSTM Error: {e}")

--- Model Evaluation ---
Prophet MAE:      13.01
Prophet Accuracy: 73.45%
LSTM MAE:         5.73
LSTM Accuracy:    88.45%
